In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

url = 'https://raw.githubusercontent.com/4GeeksAcademy/naive-bayes-project-tutorial/main/playstore_reviews.csv'
df = pd.read_csv(url)
df.shape
df.head(10)

,package_name,review,polarity
0,com.facebook.katana,privacy at least put some option appear offli...,0
1,com.facebook.katana,"messenger issues ever since the last update, ...",0
2,com.facebook.katana,profile any time my wife or anybody has more ...,0
3,com.facebook.katana,the new features suck for those of us who don...,0
4,com.facebook.katana,forced reload on uploading pic on replying co...,0
5,com.facebook.katana,idk i can't edit my posts? things such as my ...,0
6,com.facebook.katana,major flaws constant updates and always getti...,0
7,com.facebook.katana,video issues since i was forced into this upd...,0
8,com.facebook.katana,this update completely destroyed my facebook...,0
9,com.facebook.katana,"posting issues for the last week, there's bee...",0


In [11]:
df.isnull().sum()
df['polarity'].value_counts()

polarity
0    584
1    307
Name: count, dtype: int64

In [12]:
df[['review', 'polarity']].sample(5, random_state=42).to_string()

"                                                                                                                                                                                       review  polarity\n709           love/hate has bug and security issues. i tried to report that facebook and google plus have security issues and it wouldn't allow me to do so! well i just did didn't i! ......         0\n439                           whatsapp i use this app now that blackberry messenger has basically gone away. my friends & family live all over the world. this really helps keep us in touch!         1\n840                                                                                                                                                                  usefully verry  nice app         1\n720   fonts why in the heck is this thing analysing my fonts??? not really quick browsing when i have to wait 5minutes for the fonts to load. are you asking my opinion? avoid this. terrible      

In [13]:
df = df.drop(columns=['package_name'])
df = df.dropna(subset=['review'])

df['review'] = df['review'].str.strip().str.lower()
df.shape
df.head()

,review,polarity
0,privacy at least put some option appear offlin...,0
1,"messenger issues ever since the last update, i...",0
2,profile any time my wife or anybody has more t...,0
3,the new features suck for those of us who don'...,0
4,forced reload on uploading pic on replying com...,0


In [14]:
X = df['review']
y = df['polarity']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

vec_model = CountVectorizer(stop_words='english')
X_train_vec = vec_model.fit_transform(X_train).toarray()
X_test_vec  = vec_model.transform(X_test).toarray()

X_train_vec.shape
X_test_vec.shape
len(vec_model.vocabulary_)

3272

In [15]:
models = {
    'GaussianNB':    GaussianNB(),
    'MultinomialNB': MultinomialNB(),
    'BernoulliNB':   BernoulliNB()
}

results = {}
for name, model in models.items():
    model.fit(X_train_vec, y_train)
    preds = model.predict(X_test_vec)
    acc = accuracy_score(y_test, preds)
    results[name] = acc
    print(f'--- {name} ---')
    print(f'Accuracy: {acc:.4f}')
    print(classification_report(y_test, preds, target_names=['Negative','Positive']))
    print()

--- GaussianNB ---
Accuracy: 0.8156
              precision    recall  f1-score   support

    Negative       0.84      0.89      0.86       117
    Positive       0.76      0.68      0.72        62

    accuracy                           0.82       179
   macro avg       0.80      0.78      0.79       179
weighted avg       0.81      0.82      0.81       179


--- MultinomialNB ---
Accuracy: 0.8547
              precision    recall  f1-score   support

    Negative       0.84      0.96      0.90       117
    Positive       0.89      0.66      0.76        62

    accuracy                           0.85       179
   macro avg       0.87      0.81      0.83       179
weighted avg       0.86      0.85      0.85       179


--- BernoulliNB ---
Accuracy: 0.7821
              precision    recall  f1-score   support

    Negative       0.76      0.97      0.85       117
    Positive       0.87      0.44      0.58        62

    accuracy                           0.78       179
   macro avg  

In [16]:
best_nb_name = max(results, key=results.get)
print(f'Best Naive Bayes model: {best_nb_name} with accuracy {results[best_nb_name]:.4f}')

Best Naive Bayes model: MultinomialNB with accuracy 0.8547


In [18]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_vec, y_train)
rf_preds = rf_model.predict(X_test_vec)
rf_acc = accuracy_score(y_test, rf_preds)

print(f'Random Forest Accuracy: {rf_acc:.4f}')
print(classification_report(y_test, rf_preds, target_names=['Negative','Positive']))

print(f'\nBest NB ({best_nb_name}): {results[best_nb_name]:.4f}')
print(f'Random Forest:          {rf_acc:.4f}')
if rf_acc > results[best_nb_name]:
    print(' Random Forest IMPROVED over Naive Bayes!')
else:
    print(' Naive Bayes holds its ground  a common result on small text datasets.')

Random Forest Accuracy: 0.8212
              precision    recall  f1-score   support

    Negative       0.83      0.91      0.87       117
    Positive       0.79      0.66      0.72        62

    accuracy                           0.82       179
   macro avg       0.81      0.78      0.79       179
weighted avg       0.82      0.82      0.82       179


Best NB (MultinomialNB): 0.8547
Random Forest:          0.8212
 Naive Bayes holds its ground  a common result on small text datasets.


In [ ]:
os.makedirs('./models', exist_ok=True)
all_results = dict(results)  
all_results['RandomForest'] = rf_acc
best_overall = max(all_results, key=all_results.get)

best_model_obj = models.get(best_overall, rf_model)

with open('./models/best_model.pkl', 'wb') as f:
    pickle.dump(best_model_obj, f)

with open('./models/vectorizer.pkl', 'wb') as f:
    pickle.dump(vec_model, f)

print(f'Saved best model: {best_overall} (accuracy={all_results[best_overall]:.4f})')


Saved best model: MultinomialNB (accuracy=0.8547)
Files saved to ./models/


In [20]:
alt_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'LinearSVC':           LinearSVC(max_iter=1000, random_state=42)
}

for name, model in alt_models.items():
    model.fit(X_train_vec, y_train)
    preds = model.predict(X_test_vec)
    acc = accuracy_score(y_test, preds)
    all_results[name] = acc
    print(f'--- {name} ---')
    print(f'Accuracy: {acc:.4f}')
    print(classification_report(y_test, preds, target_names=['Negative','Positive']))
    print()

--- Logistic Regression ---
Accuracy: 0.8324
              precision    recall  f1-score   support

    Negative       0.86      0.89      0.87       117
    Positive       0.78      0.73      0.75        62

    accuracy                           0.83       179
   macro avg       0.82      0.81      0.81       179
weighted avg       0.83      0.83      0.83       179


--- LinearSVC ---
Accuracy: 0.8101
              precision    recall  f1-score   support

    Negative       0.87      0.83      0.85       117
    Positive       0.71      0.77      0.74        62

    accuracy                           0.81       179
   macro avg       0.79      0.80      0.79       179
weighted avg       0.82      0.81      0.81       179




In [23]:
summary = pd.DataFrame(list(all_results.items()), columns=['Model', 'Accuracy'])
summary = summary.sort_values('Accuracy', ascending=False).reset_index(drop=True)
summary['Accuracy'] = summary['Accuracy'].round(4)

print(summary.to_string(index=False))
print(f'\n Winner: {summary.iloc[0]["Model"]} with {summary.iloc[0]["Accuracy"]} accuracy')

              Model  Accuracy
      MultinomialNB    0.8547
Logistic Regression    0.8324
       RandomForest    0.8212
         GaussianNB    0.8156
          LinearSVC    0.8101
        BernoulliNB    0.7821

 Winner: MultinomialNB with 0.8547 accuracy


## Conclusion
In this project we built a sentiment classifier for Google Play reviews using three Naive Bayes variants (GaussianNB, MultinomialNB, BernoulliNB), where **MultinomialNB performed best** among them since it is specifically designed for word-count data. We then tested **Random Forest**, **Logistic Regression**, and **LinearSVC** as alternatives — linear models like Logistic Regression and LinearSVC tend to outperform Naive Bayes on text tasks because they don't assume word independence, making them better at capturing real language patterns. Overall, Naive Bayes remains a solid and fast baseline for text classification, but for higher accuracy, linear discriminative models are the recommended choice.